In [ ]:
import os
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
modulePath = os.path.abspath(os.path.join('/home/liangmj/repo/scripts/cases'))
if modulePath not in sys.path:
    sys.path.append(modulePath)
from common.unitConverter import UnitConverter as uc

# plt.rcParams.update({
#     "font.size" : 18,
#     "font.family": "serif",
#     "font.serif": ["Times New Roman"],
#     "mathtext.fontset": "stix",
# })


In [ ]:
folderVec = [
    # "/home/liangmj/repo/LBM-Program/amrlbm/bin/acoustic_Re150/cylinderRe150_cml_m02_25dx_test"
    # "/home/liangmj/repo/LBM-Program/amrlbm/bin/acoustic_Re150/cylinderRe150_cml_m02_25dx_test_2",
    "/home/liangmj/repo/LBM-Program/amrlbm/bin/acoustic_Re150/cylinderRe150_cml_m02_25dx_test_2_OG",
    # "/home/liangmj/repo/LBM-Program/amrlbm/bin/acoustic_Re150/cylinderRe150_cml_m02_25dx_test_3"
    # "/home/liangmj/repo/LBM-Program/amrlbm/bin/acoustic_Re150/cylinderRe150_cml_m02_25dx_test_2_75D"
]

farNearFolder = [
    ["farfield2", "probe3"],
    ["farfield2", "probe3"],
]

lblVec = [
    # "LBM Cumulant D/dx=25 test N=8",
    # "LBM Cumulant D/dx=25 test N=1",
    "LBM Cumulant D/dx=25 test N=8",
]

DPhy = 1.0

DLbVec = [
    25,
    25,
]

rhoPhyVec = [
    1.204,
    1.204,
]
gamma = 1.0

U0PhyVec = [
    68.0,
    68.0,
]

stepRg = [
    [60000, 100000],
    [60000, 100000],
]

# compareNearFar = "near"
# compareNearFar = "far"
compareNearFar = "both"

dxVec = [ DPhy/DLb for DLb in DLbVec ]
unitConverterVec = [ uc(dx, rhoPhy, gamma) for dx, rhoPhy in zip(dxVec, rhoPhyVec) ]
pres0PhyVec = [ unitConverter.lb_to_phys_pressure(1.0 * unitConverter.cs2_lb) for unitConverter in unitConverterVec ]



In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

def get_probe_coords(filePath):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordsStr = header.split('(')[1].split(')')[0].split(',')
        x = float(coordsStr[0])
        y = float(coordsStr[1])
        z = float(coordsStr[2])
    return x,y,z

def preFreqDomain_to_splFreqDomain(preFreqDomain, refPressure):
    spl = 20 * np.log10(np.abs(preFreqDomain) / np.sqrt(2) / refPressure)
    return spl

def splFreqDomain_to_psdFreqDomain(splFreqDomain, deltaFreq):
    psd = splFreqDomain - 10 * np.log10(deltaFreq)
    return psd

def preFreqDomain_to_psdFreqDomain(preFreqDomain, refPressure, deltaFreq):
    spl = preFreqDomain_to_splFreqDomain(preFreqDomain, refPressure)
    psd = splFreqDomain_to_psdFreqDomain(spl, deltaFreq)
    return psd

def 

In [ ]:
farRadVecVec = []
farPresRMSVecVec = []

nearRadVecVec = []
nearPresRMSVecVec = []

for case in range(0, len(folderVec)):
    if compareNearFar == "far" or compareNearFar == "both":
        print("wow far")
        farFileList = sorted(glob.glob(f"{folderVec[case]}/microphones/{farNearFolder[case][0]}/microphone*.txt"))
        farRadVec = []
        farPresFluctuateRMSVec = []
        for iFar in range(0, len(farFileList)):
            x, y, z = get_probe_coords(farFileList[iFar])
            radians = np.arctan2(y, x) % (2 * np.pi)
            farRadVec.append(radians)
            stepCol = read_col_probe(farFileList[iFar], 0, 2)
            stepCol = stepCol - stepCol[0]
            pressureCol = read_col_probe(farFileList[iFar], 2, 2)
            idxStart = int(np.abs(stepCol - stepRg[case][0]).argmin())
            # idxEnd = int(np.abs(stepCol - stepRg[case][1]).argmin())
            # idxEnd = len(stepCol) - 1
            deltaPres = pressureCol - pres0PhyVec[case]
            deltaPresTilde = deltaPres[idxStart:] - np.mean(deltaPres[idxStart:])
            deltaPresTildeRMS = np.sqrt(np.mean(deltaPresTilde**2))
            # farPresFluctuateRMSVec.append(deltaPresTildeRMS) # physical unit
            farPresFluctuateRMSVec.append(unitConverterVec[case].phys_to_lb_pressure(deltaPresTildeRMS)) # lb unit
        farRadVecVec.append(farRadVec)
        farPresRMSVecVec.append(farPresFluctuateRMSVec)

    if compareNearFar == "near" or compareNearFar == "both":
        print("wow near")
        nearRadVec = []
        nearPresFluctuateRMSVec = []
        nearFileList = sorted(glob.glob(f"{folderVec[case]}/probes/{farNearFolder[case][1]}/probe*.txt"))
        for iNear in range(0, len(nearFileList)):
            x, y, z = get_probe_coords(nearFileList[iNear])
            radians = np.arctan2(y, x) % (2 * np.pi)
            nearRadVec.append(radians)
            stepCol = read_col_probe(nearFileList[iNear], 0, 2)
            rhoCol = read_col_probe(nearFileList[iNear], 2, 2)
            idxStart = int(np.abs(stepCol - stepRg[case][0]).argmin())
            deltaPres = (rhoCol - 1.0) * unitConverterVec[case].cs2_lb
            deltaPresTilde = deltaPres[idxStart:] - np.mean(deltaPres[idxStart:])
            deltaPresTildeRMS = np.sqrt(np.mean(deltaPresTilde**2)) # lb unit
            nearPresFluctuateRMSVec.append(deltaPresTildeRMS)
        nearRadVecVec.append(nearRadVec)
        nearPresRMSVecVec.append(nearPresFluctuateRMSVec)
    print(farRadVecVec)
    print(nearRadVecVec)



In [ ]:
fig1, ax1 = plt.subplots(1, 1, figsize=(8, 8), facecolor='w', edgecolor='w', subplot_kw={'projection': 'polar'})

for case in range(0, len(folderVec)):
    if compareNearFar == "far" or compareNearFar == "both":
        ax1.plot(farRadVecVec[case], farPresRMSVecVec[case], label=f"{lblVec[case]} FWH")
    if compareNearFar == "near" or compareNearFar == "both":
        print(min(nearPresRMSVecVec[case]))
        degArray = np.degrees(nearRadVecVec[case])
        angle = 58
        mask = ~((degArray <= angle) | (degArray >= (360-angle)))
        # ax1.plot(nearRadVecVec[case], nearPresRMSVecVec[case], label=f"{lblVec} nearfield")
        ax1.plot(np.array(nearRadVecVec[case])[mask], np.array(nearPresRMSVecVec[case])[mask], label=f"{lblVec} nearfield")
